# SmartCart Day 1 - Acquire & Index

We turn a pile of product photos into two reusable assets: a clean **labels table** and a searchable **catalog gallery** built on frozen DINOv2 embeddings. Everything we make here is saved to the cross-day *bundle* so Day 2 can pick it up.

In [1]:
# 1) Runtime setup
%pip install -q timm

import os

# The cross-day bundle lives in a local folder (was a Google Drive mount on Colab).
# Day 2 (day_2/Day2_localize.ipynb) reads/writes this same path for continuity.
BUNDLE_DIR = os.path.expanduser('~/SmartCart_bundle')


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


### Utilities Toolkit

This hidden cell defines the helper functions used below. Run it once after setup.

In [2]:
from __future__ import annotations
import json
import os
import pathlib
import subprocess
import numpy as np
import pandas as pd
HERE = pathlib.Path.cwd()  # embedded in-notebook: no __file__, anchor on the working dir
GROCERY_DATASET_URL = 'https://github.com/marcusklasson/GroceryStoreDataset'

class Bundle:
    """Small local folder that carries artifacts from one day to the next."""

    def __init__(self, root: str):
        self.root = pathlib.Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.manifest = {'version': 1, 'class_list': [], 'artifacts': {}}

    def put_table(self, name, df: pd.DataFrame):
        df.to_csv(self.root / name, index=False)
        self._note(name)

    def get_table(self, name) -> pd.DataFrame:
        return pd.read_csv(self.root / name)

    def put_array(self, name, arr: np.ndarray):
        np.save(self.root / name, arr)
        self._note(name)

    def get_array(self, name) -> np.ndarray:
        p = self.root / name
        return np.load(p if p.suffix == '.npy' else p.with_suffix('.npy'))

    def _note(self, name):
        self.manifest['artifacts'][name] = True

    def save(self):
        (self.root / 'manifest.json').write_text(json.dumps(self.manifest, indent=2))

    def load(self):
        p = self.root / 'manifest.json'
        if p.exists():
            self.manifest = json.loads(p.read_text())
        return self

def open_bundle(drive_dir=None) -> Bundle:
    """Open the cross-day local bundle. If it is new, start with an empty manifest."""
    return Bundle(drive_dir or os.path.expanduser('~/SmartCart_bundle')).load()

def save_bundle(b: Bundle):
    b.save()
    print(f'[bundle] saved -> {b.root}')

def open_grocery_dataset():
    """Clone or reuse the real GroceryStoreDataset and return (tier, root_dir)."""
    root = HERE / 'GroceryStoreDataset'
    if (root / 'dataset' / 'classes.csv').exists():
        print('using cached GroceryStoreDataset:', root)
    else:
        print('cloning GroceryStoreDataset from:', GROCERY_DATASET_URL)
        subprocess.run(['git', 'clone', '--depth', '1', GROCERY_DATASET_URL, str(root)], check=True)
    if not (root / 'dataset' / 'classes.csv').exists():
        raise RuntimeError('GroceryStoreDataset clone is incomplete. Check network access and rerun.')
    print('using data tier: github')
    print('data root:', root)
    print('OK: using real GroceryStoreDataset images.')
    return ('github', str(root))

def open_sg_dataset(sg_root=None):
    """SG innovation swap for open_grocery_dataset(): point at our own Singapore
    catalog (sg_data/) instead of the course-default GroceryStoreDataset. Returns
    the same (tier, root_dir) shape so downstream code doesn't need to branch."""
    root = pathlib.Path(sg_root or (HERE.parent / 'sg_data'))
    if not (root / 'labels.csv').exists():
        raise RuntimeError(f'sg_data not found at {root} (expected labels.csv). '
                            f'Check the path or run Week4/sg_data/collect_sg_groceries.py first.')
    print('using data tier: sg_data (Singapore catalog)')
    print('data root:', root)
    print('OK: using real FairPrice-sourced Singapore product images.')
    return ('sg_data', str(root))

def list_images(root, per_class=None):
    """Return a DataFrame with columns path,coarse,fine.
    Handles two layouts: GroceryStoreDataset's train/val/test split folders, and
    sg_data's flat labels.csv (path,coarse,fine) — detected by which one exists."""
    root = pathlib.Path(root)
    labels_csv = root / 'labels.csv'
    if labels_csv.exists():
        df = pd.read_csv(labels_csv)
        df['path'] = df['path'].map(lambda p: p if pathlib.Path(p).is_absolute() else str((root / p).resolve()))
    else:
        rows = []
        for split in ('train', 'val', 'test'):
            for p in (root / 'dataset' / split).rglob('*.jpg'):
                rows.append({'path': str(p), 'coarse': p.parent.parent.name, 'fine': p.parent.name})
        df = pd.DataFrame(rows)
    if per_class and len(df):
        df = df.groupby('fine', group_keys=False).head(per_class).reset_index(drop=True)
    return df

def list_gallery(root):
    """Return [(iconic_path, fine), ...] — one reference image per class.
    GroceryStoreDataset: from dataset/classes.csv. sg_data: from sg_dataset/iconic/<Fine>.jpg."""
    root = pathlib.Path(root)
    sg_iconic = root / 'sg_dataset' / 'iconic'
    if sg_iconic.exists():
        return [(str(p), p.stem) for p in sorted(sg_iconic.glob('*.jpg'))]
    df = pd.read_csv(root / 'dataset' / 'classes.csv')
    name_col = next((c for c in df.columns if c.lower().startswith('class name')))
    icon_col = next((c for c in df.columns if 'iconic' in c.lower()))
    gallery = []
    for _, r in df.iterrows():
        p = root / 'dataset' / str(r[icon_col]).lstrip('/')
        if p.exists():
            gallery.append((str(p), str(r[name_col])))
    return gallery

def load_backbone(name='vit_small_patch14_dinov2.lvd142m', device='cpu'):
    """Frozen DINOv2-small feature extractor."""
    import timm
    m = timm.create_model(name, pretrained=True, num_classes=0, dynamic_img_size=True).eval().to(device)
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def extract_features(model, batches, device=None) -> np.ndarray:
    """Run image batches through a frozen backbone and return numpy features."""
    import torch
    if device is None:
        try:
            device = next(model.parameters()).device
        except (AttributeError, StopIteration):
            device = 'cpu'
    outs = []
    with torch.no_grad():
        for xb in batches:
            y = model(xb.to(device))
            outs.append(y.detach().cpu().numpy().astype('float32'))
    return np.concatenate(outs, 0)

class EmbeddingIndex:
    """Cosine nearest-neighbor index over catalog/gallery embeddings."""

    def __init__(self):
        self._nn = None
        self.meta = []

    def build(self, feats: np.ndarray, meta: list[dict]):
        from sklearn.neighbors import NearestNeighbors
        f = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-08)
        self._nn = NearestNeighbors(n_neighbors=min(10, len(f)), metric='cosine').fit(f)
        self.meta = meta
        self._feats = f
        return self

    def query(self, q: np.ndarray, k=5):
        qn = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-08)
        dist, idx = self._nn.kneighbors(qn, n_neighbors=min(k, len(self.meta)))
        return [[{**self.meta[j], 'score': float(1 - d)} for d, j in zip(drow, irow)] for drow, irow in zip(dist, idx)]

    def save(self, b: Bundle, prefix='gallery'):
        b.put_array(f'{prefix}_index.npy', self._feats)
        b.put_table(f'{prefix}_meta.csv', pd.DataFrame(self.meta))

    @classmethod
    def load(cls, b: Bundle, prefix='gallery'):
        feats = b.get_array(f'{prefix}_index.npy')
        meta = b.get_table(f'{prefix}_meta.csv').to_dict('records')
        return cls().build(feats, meta)
# --- helpers are now available as plain functions/classes in this notebook ---
print('SmartCart toolkit ready')

SmartCart toolkit ready


In [3]:
# 2) Load the cross-day bundle
# The bundle stores artifacts we create during the week: labels, indexes, weights, ONNX files.
# It is NOT the image dataset. Images are loaded in the next data-source cell.
b = open_bundle(BUNDLE_DIR)
print('bundle:', b.root)
print('artifacts:', list(b.manifest.get('artifacts', {})))


bundle: /home/jonyling/SmartCart_bundle
artifacts: ['sample_scene.jpg', 'detector.pt', 'crops_manifest.csv', 'gallery_index.npy', 'gallery_meta.csv', 'catalog_prices.csv', 'labels.csv', 'head.pt', 'per_class_metrics.csv', 'error_report.md', 'head_v2.pt', 'lift_table.csv']


In [4]:
# 3) Select the image data source
# SG innovation swap (D1): use our own Singapore catalog instead of the course
# default. Both return the same (tier, root) shape, so nothing downstream changes.
USE_SG_CATALOG = True
if USE_SG_CATALOG:
    tier, root = open_sg_dataset()
else:
    tier, root = open_grocery_dataset()


using data tier: sg_data (Singapore catalog)
data root: /home/jonyling/SNAIC/Week4/sg_data
OK: using real FairPrice-sourced Singapore product images.


## Curate

**What:** List every GroceryStoreDataset product image into a `path,fine` table.

**Why:** For the real dataset, labels come from the folder structure. This labels table is the contract every later leg reads.

**Watch for:** Confirm the previous cell printed `using data tier: github` before continuing.

In [5]:
import pandas as pd
# Read image paths and labels from whichever tier was selected above.
# SC_PER_CLASS keeps the real dataset small enough for a fast class run.
per_class = int(os.environ.get('SC_PER_CLASS','40'))
df = list_images(root, per_class=per_class)
assert len(df) > 0, 'No images found in the selected data source. Re-run the data-source cell.'
print('images:', len(df))
print('fine classes:', df.fine.nunique())
print('label source:', 'sg_data labels.csv (folder-derived, FairPrice scrape)' if tier == 'sg_data'
      else 'folder names from GroceryStoreDataset')
print(df.fine.value_counts().head(12))


images: 998
fine classes: 25
label source: sg_data labels.csv (folder-derived, FairPrice scrape)
fine
Ayam-Brand-Sardines    40
Bananas                40
Camel-Nuts             40
Chye-Sim               40
FN-Seasons             40
Ice-Mountain-Water     40
KCT-Chilli-Sauce       40
Kailan                 40
Kaya-Spread            40
Meiji-Fresh-Milk       40
Khong-Guan-Biscuits    40
Koka-Noodles           40
Name: count, dtype: int64


## Optional annotation discussion

**What:** Discuss how manual annotation would work, but do not load separate demo data.

**Why:** GroceryStoreDataset is already organized by labeled folders, so this lab does not need Label Studio to produce training labels.

**Watch for:** You should treat `df` above as the source of truth for Day 1 labels.

In [6]:
print('No manual annotation is required for this lab.')
print('Training labels come from GroceryStoreDataset folder names in df.fine.')
print('Optional class discussion: what quality issues would you audit before trusting these labels?')


No manual annotation is required for this lab.
Training labels come from GroceryStoreDataset folder names in df.fine.
Optional class discussion: what quality issues would you audit before trusting these labels?


## Embeddings

**What:** Run every image through a frozen DINOv2-small backbone to get 384-d feature vectors.

**Why:** Frozen features are cheap, stable, and good enough that a tiny head (Day 3) can classify on top.

**Watch for:** First call downloads the backbone weights. Feed 224x224 - the loader interpolates the pos-embeds.

In [7]:
import torch, torchvision.transforms.v2 as T
from PIL import Image
import numpy as np
# Shared image preprocessing for DINOv2.
TF = T.Compose([T.ToImage(), T.Resize((224,224)), T.ToDtype(torch.float32, scale=True),
                T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def feats_of(paths, model, bs=16):
    """Embed image files in small batches and return an (N, D) numpy array."""
    out=[]
    for i in range(0,len(paths),bs):
        xs=torch.stack([TF(Image.open(p).convert('RGB')) for p in paths[i:i+bs]])
        out.append(extract_features(model,[xs]))
    return np.concatenate(out,0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device:', device)
model = load_backbone(device=device)
# One feature vector per product image.
feats = feats_of(list(df.path), model)
print('feature matrix:', feats.shape)  # (N, 384)


using device: cuda


/home/jonyling/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/jonyling/miniconda3/lib/python3.13/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


feature matrix: (998, 384)


## Build the catalog gallery + search

**What:** Embed the one iconic image per class, index it, and query a real photo against it.

**Why:** This is visual product search: a new photo finds its nearest catalog entry by cosine similarity.

**Watch for:** Predict first! Before running the query, guess: will photo[0] retrieve its own class as the top hit?

In [8]:
# The gallery has one reference/catalog image per class.
gallery = list_gallery(root)  # [(iconic_path, fine), ...]
icon_paths = [p for p, _ in gallery]
gfeats = feats_of(icon_paths, model)
gmeta = [{'fine': f, 'iconic': p} for p, f in gallery]
idx = EmbeddingIndex().build(gfeats, gmeta)
# Query the first product photo against the catalog.
print('top-3 for', df.fine.iloc[0], '->', idx.query(feats[0:1], k=3)[0])
idx.save(b)  # writes gallery_index.npy + gallery_meta.csv


top-3 for Ayam-Brand-Sardines -> [{'fine': 'Ayam-Brand-Sardines', 'iconic': '/home/jonyling/SNAIC/Week4/sg_data/sg_dataset/iconic/Ayam-Brand-Sardines.jpg', 'score': 1.0}, {'fine': 'Milo-Powder', 'iconic': '/home/jonyling/SNAIC/Week4/sg_data/sg_dataset/iconic/Milo-Powder.jpg', 'score': 0.6701323390007019}, {'fine': 'Maggi-Curry-Noodles', 'iconic': '/home/jonyling/SNAIC/Week4/sg_data/sg_dataset/iconic/Maggi-Curry-Noodles.jpg', 'score': 0.5491338968276978}]


## Create the price catalog

**What:** Create `catalog_prices.csv`, the small business table used by the final checkout demo.

**Why:** The dataset gives product labels, not real shop prices. We add editable demo prices so Day 5 can compute a basket total.

**Watch for:** Students may edit this table later to use local currency, promotions, tax, or richer product metadata.

In [9]:
from pathlib import Path
if tier == 'sg_data':
    # D1 innovation: REAL prices, not demo prices. These are median as-sold SGD
    # prices scraped from FairPrice Online per SKU (see sg_data/sg_products_raw.csv
    # for full provenance). Nothing synthetic to fabricate here — just load and save.
    catalog = pd.read_csv(Path(root) / 'catalog_prices.csv')
    print('REAL FairPrice SGD prices (median as-sold, scraped ' 
          + str(pd.read_csv(Path(root) / "sg_products_raw.csv").scrape_date.max()) + '):')
else:
    # Demo price catalog. These are illustrative, not from GroceryStoreDataset.
    if 'coarse' not in df.columns:
        # Older cached runs may have only path,fine. Recover coarse from the folder above fine.
        df['coarse'] = df.path.map(lambda p: Path(p).parent.parent.name)
    catalog = (df[['coarse','fine']].drop_duplicates().sort_values('fine').reset_index(drop=True))
    base_price = {'Fruit': 0.80, 'Vegetables': 0.70, 'Packages': 1.50}
    catalog['unit_price'] = [round(base_price.get(row.coarse, 1.00) + 0.05*(i % 12), 2)
                             for i, row in catalog.iterrows()]
    catalog['currency'] = 'USD'
    catalog['unit'] = 'each'
    catalog = catalog[['fine','coarse','unit_price','currency','unit']]
print(catalog.head(12).to_string(index=False))
b.put_table('catalog_prices.csv', catalog)


REAL FairPrice SGD prices (median as-sold, scraped 2026-07-09):
                fine     coarse  unit_price currency unit
 Ayam-Brand-Sardines     Canned        3.73      SGD each
             Bananas      Fruit        4.55      SGD each
          Camel-Nuts     Snacks        6.66      SGD each
            Chye-Sim Vegetables        2.30      SGD each
          FN-Seasons  Beverages        3.20      SGD each
Gardenia-White-Bread     Bakery        3.00      SGD each
  Ice-Mountain-Water  Beverages        8.96      SGD each
    KCT-Chilli-Sauce     Sauces        2.63      SGD each
              Kailan Vegetables        2.20      SGD each
         Kaya-Spread    Spreads        3.85      SGD each
 Khong-Guan-Biscuits     Snacks        2.70      SGD each
        Koka-Noodles    Noodles        2.40      SGD each


## Save + carry forward

**What:** Persist the labels table, price catalog, and class list, then save the bundle.

**Why:** Day 2 reads `labels.csv`; Day 5 reads `catalog_prices.csv` to turn predictions into a receipt.

**Watch for:** Confirm the carry-forward print lists labels.csv, catalog_prices.csv, and the gallery arrays.

In [10]:
b.put_table('labels.csv', df)
b.manifest['class_list'] = sorted(df.fine.unique().tolist())
save_bundle(b)
print('\u25b6 Carries forward: labels.csv + catalog_prices.csv + gallery_index')


[bundle] saved -> /home/jonyling/SmartCart_bundle
▶ Carries forward: labels.csv + catalog_prices.csv + gallery_index
